# Research Project Template
## optlab-research | Northeastern University Options Club

**How to use this template:**
1. Copy this notebook to `notebooks/your_name/your_project.ipynb`
2. Replace the hypothesis, signal name, and date range below
3. Run all cells top-to-bottom
4. Customize the report title and save

**Time to first result: ~30 minutes**

## Step 0: State Your Hypothesis
*Fill this in before writing any code.*

**Research question:** Does the low-volatility anomaly survive transaction costs in the Russell 1000?

**Signal:** `beta_60m` — 60-month rolling market beta. Low-beta stocks should outperform on a risk-adjusted basis (Frazzini-Pedersen 2014).

**Expected result:** Long low-beta (Q1), short high-beta (Q5). Expect positive Sharpe before costs, degradation at 20+ bps.

**Null hypothesis:** There is no statistically significant alpha after controlling for FF6 factors.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import optlab_research.workbench as wb

# ── USER SETTINGS ── change these for your project ──────────────────────
SIGNAL = "beta_60m" # signal name from signals.yaml
UNIVERSE = "russell1000" # russell3000, russell1000, liquid_500
START = "2019-01-01" # backtest start date
END = "2023-12-31" # backtest end date
LONG_Q = 1 # quantile to go long (1=lowest beta)
SHORT_Q = 5 # quantile to go short (5=highest beta)
TCOST_BPS = 10 # transaction cost assumption
REPORT_TITLE = "Low Volatility / BAB Factor"
REPORT_SUB = f"{UNIVERSE.title()} | {START[:4]}–{END[:4]} | tcost={TCOST_BPS}bps"
# ─────────────────────────────────────────────────────────────────────────

print("Settings loaded:")
print(f" Signal: {SIGNAL}")
print(f" Universe: {UNIVERSE}")
print(f" Period: {START} → {END}")
print(f" L/S: Q{LONG_Q} / Q{SHORT_Q}")
print(f" T-cost: {TCOST_BPS} bps")

## Step 1: Validate the Signal (Single Date Cross-Section)

In [ ]:
CHECK_DATE = END # use end date for validation

with wb.open() as con:
 sig = wb.signal(SIGNAL, CHECK_DATE, con=con, n_quantiles=5)

print(f"Signal: {SIGNAL} as of {CHECK_DATE}")
print(f"Valid rows: {sig.drop_nulls().shape[0]} / {sig.shape[0]}")
print(f"Coverage: {sig.drop_nulls().shape[0] / sig.shape[0]:.1%}")
print()
print(sig.head(10))

## Step 2: Run the Backtest

In [ ]:
with wb.open() as con:
 bt = wb.backtest(
 signal_name = SIGNAL,
 start = START,
 end = END,
 universe = UNIVERSE,
 long_quantile = LONG_Q,
 short_quantile = SHORT_Q,
 tcost_bps = TCOST_BPS,
 con = con,
 )

s = bt.summary()
print(f"Period: {s.get('start_date','N/A')} → {s.get('end_date','N/A')}")
print(f"Ann. Return L/S: {s.get('ann_return_ls', 0):.2%}")
print(f"Sharpe L/S: {s.get('sharpe_ls', 0):.3f}")
print(f"Max Drawdown: {s.get('max_drawdown_ls', 0):.2%}")
print(f"Win Rate: {s.get('win_rate_ls', 0):.1%}")
print(f"Avg Turnover: {s.get('avg_monthly_turnover', 0):.1%}")

## Step 3: Factor Attribution

In [ ]:
with wb.open() as con:
 attr = wb.attribution(bt, model="ff6", con=con)

print("FF6 Factor Attribution")
print(f"Alpha: {attr.get('alpha', 0):.4f} (t = {attr.get('t_stats', {}).get('alpha', 0):.2f})")
print(f"R²: {attr.get('r_squared', 0):.3f}")
print()
print("Factor loadings:")
for factor, beta in attr.get('betas', {}).items():
 t = attr.get('t_stats', {}).get(factor, 0)
 print(f" {factor:<8} β = {beta:+.3f} (t = {t:.2f})")

## Step 4: Generate Report

In [ ]:
os.makedirs("../../outputs", exist_ok=True)
report_path = f"../../outputs/{SIGNAL}_report.html"

rpt = wb.report(bt, title=REPORT_TITLE, subtitle=REPORT_SUB)
rpt.save(report_path)
print(f"Report saved: {report_path}")
print("Open in a browser to view.")

## Step 5: State Your Conclusion

*Fill this in after reviewing the results.*

**Finding:** [Describe what the numbers show]

**Interpretation:** [Why do you think this result occurred?]

**Limitations:** [What would make this analysis stronger?]

**Next steps:** [What would you test next?]

---
*This template produced using optlab-research v0.1 · Northeastern University Options Club · Fall 2026*